# Gradients Real Training Demo

This notebook trains a small Qwen base model on the Gradients PubMedQA normalized training set, then compares answers from the base model and the trained model on a few held-out test prompts.

The aim is that the trained model will provide factually correct medical data and improved evidence over the base model.

You only need a Gradients API key to launch training. For loading a private trained model from Hugging Face, set `HF_TOKEN` in your environment before running the inference cells.

## Install Packages

Run this once. `%pip` installs into the same Python environment used by this notebook.

In [ ]:
%pip install -q --upgrade torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1
%pip install -q --upgrade gradientsio==0.1.1 datasets==2.21.0 transformers==4.57.1 accelerate==1.11.0 peft==0.17.1 huggingface_hub==0.36.0 pandas==2.2.3

## Setup

Enter your Gradients API key when prompted. The defaults use the normalized PubMedQA train/test datasets, `Qwen/Qwen2.5-3B`, and a 2 hour training run.

In [ ]:
import gc
import os
import time
from getpass import getpass

import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import Markdown
from IPython.display import display
from gradientsio import GradientsClient
from gradientsio import TaskType

MODEL_ID = "Qwen/Qwen2.5-3B"
TRAIN_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Train"
TEST_DATASET = "gradients-io-tournaments/PubMedQA-Normalized-Test"
HOURS_TO_TRAIN = 2
RESULT_MODEL_NAME = "pubmedqa-qwen2-5-3b-gradients-demo"
SAMPLE_SIZE = 5
MAX_NEW_TOKENS = 96
REPETITION_PENALTY = 1.12
NUM_BEAMS = 4
POLL_INTERVAL_SECONDS = 300
# Inference uses NVIDIA CUDA only. Set False only if you intentionally want CPU (slow).
REQUIRE_CUDA = True

if not os.getenv("GRADIENTS_API_KEY"):
    os.environ["GRADIENTS_API_KEY"] = getpass("Gradients API key: ").strip()

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
client = GradientsClient()

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "REQUIRE_CUDA is True but PyTorch does not see CUDA (torch.cuda.is_available() is False). "
        "Install a CUDA-enabled PyTorch build from https://pytorch.org for your driver/CUDA version; "
        "CPU-only pip wheels will never use an NVIDIA GPU. Confirm `nvidia-smi` works in the same environment."
    )
if torch.cuda.is_available():
    print(
        f"CUDA OK — {torch.cuda.get_device_name(0)} | torch {torch.__version__} | cuda {torch.version.cuda}"
    )
elif not REQUIRE_CUDA:
    print("Warning: CUDA unavailable; inference will use CPU (slow).")

print("Setup complete.")
print(f"Base model: {MODEL_ID}")
print(f"Train dataset: {TRAIN_DATASET}")
print(f"Test dataset: {TEST_DATASET}")

## Start Training

This creates an instruct fine-tuning task on Gradients. Save the printed task ID if you close the notebook; you can paste it into the wait cell later.

In [ ]:
task = client.train(
    model=MODEL_ID,
    task_type=TaskType.INSTRUCT,
    hours=HOURS_TO_TRAIN,
    dataset=TRAIN_DATASET,
    field_instruction="instruction",
    field_input="input",
    field_output="output",
    result_model_name=RESULT_MODEL_NAME,
)

TASK_ID = task.task_id
print(f"Training task created: {TASK_ID}")
print("You can now run the next cell to wait for training to finish.")

## Wait For Training

Run this after starting training. If you already have a task ID from a previous run, set `TASK_ID` before running this cell.

In [ ]:
if "TASK_ID" not in globals() or not TASK_ID:
    TASK_ID = input("Paste an existing Gradients task ID: ").strip()

training_task = client.tasks.handle(TASK_ID)

while True:
    details = training_task.refresh()
    status = details.status
    trained_repo = details.trained_model_repository
    print(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | status={status} | trained_model_repository={trained_repo}")

    if details.is_terminal:
        break

    time.sleep(POLL_INTERVAL_SECONDS)

if not details.is_success:
    raise RuntimeError(f"Training did not finish successfully. Final status: {details.status}")

TRAINED_MODEL_REPO = details.trained_model_repository
if not TRAINED_MODEL_REPO:
    raise RuntimeError("Training succeeded, but no trained_model_repository was returned.")

print(f"Training complete: {TRAINED_MODEL_REPO}")

## Pick Test Prompts

This samples a few held-out examples from the normalized PubMedQA test dataset. Each example has an instruction, abstracts in `input`, and the expected answer in `output`.

In [ ]:
test_ds = load_dataset(TEST_DATASET, split="train")
examples = test_ds.shuffle(seed=23223).select(range(SAMPLE_SIZE))


def build_prompt(example):
    instruction = (example.get("instruction") or "").strip()
    context = (example.get("input") or "").strip()
    return (
        f"{instruction}\n\n"
        "Answer:"
    )


preview_rows = []
for index, example in enumerate(examples, start=1):
    preview_rows.append(
        {
            "example": index,
            "pubid": example.get("pubid"),
            "prompt_preview": build_prompt(example)[:500] + "...",
            "expected_answer": example.get("output"),
        }
    )

pd.DataFrame(preview_rows)

## Generation Helpers

These helpers run text generation and load either a full trained model repo or a LoRA/PEFT adapter repo. If the trained repo contains `adapter_config.json`, the notebook loads the base model, applies the adapter, and merges it for inference.

In [ ]:
from huggingface_hub import HfApi
from peft import PeftModel
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

api = HfApi(token=HF_TOKEN)


def repo_has_file(repo_id, filename):
    try:
        info = api.model_info(repo_id)
    except Exception as exc:
        raise RuntimeError(
            f"Could not read model repo {repo_id!r}. If it is private, set HF_TOKEN before running this notebook."
        ) from exc

    return any(sibling.rfilename == filename for sibling in info.siblings)


def model_kwargs():
    kwargs = {"trust_remote_code": True}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    if torch.cuda.is_available():
        kwargs["device_map"] = "auto"
        kwargs["torch_dtype"] = torch.bfloat16
        kwargs["low_cpu_mem_usage"] = True
    else:
        kwargs["torch_dtype"] = torch.float32
    return kwargs


def _require_cuda_if_configured():
    if globals().get("REQUIRE_CUDA", True) and not torch.cuda.is_available():
        raise RuntimeError(
            "Run the setup cell first with REQUIRE_CUDA=True, or install CUDA PyTorch so the GPU is visible."
        )


def load_tokenizer(repo_id):
    kwargs = {"trust_remote_code": True}
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    tokenizer = AutoTokenizer.from_pretrained(repo_id, **kwargs)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model(repo_id, *, base_model_id=None):
    _require_cuda_if_configured()
    if base_model_id and repo_has_file(repo_id, "adapter_config.json"):
        print(f"Loading base model {base_model_id} and merging LoRA adapter {repo_id}...")
        tokenizer = load_tokenizer(base_model_id)
        base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **model_kwargs())
        model = PeftModel.from_pretrained(base_model, repo_id, token=HF_TOKEN)
        model = model.merge_and_unload()
        model.eval()
        return tokenizer, model

    print(f"Loading full model {repo_id}...")
    tokenizer = load_tokenizer(repo_id)
    model = AutoModelForCausalLM.from_pretrained(repo_id, **model_kwargs())
    model.eval()
    return tokenizer, model


def generate_answer(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072)
    device = next(model.parameters()).device
    inputs = {name: value.to(device) for name, value in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id,
        "repetition_penalty": REPETITION_PENALTY,
    }
    if NUM_BEAMS > 1:
        gen_kwargs["num_beams"] = NUM_BEAMS
        gen_kwargs["early_stopping"] = True

    with torch.no_grad():
        generated = model.generate(**inputs, **gen_kwargs)

    new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def release_model(model=None):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Generation helpers ready (CUDA-only when REQUIRE_CUDA=True).")

## Run Base Model

This runs the original Qwen base model on the sampled test prompts.

In [ ]:
n_examples = len(examples)
print(f"[base] Loading {MODEL_ID!r} ({n_examples} examples) …", flush=True)
t_load = time.perf_counter()
base_tokenizer, base_model = load_model(MODEL_ID)
print(f"[base] Load finished in {time.perf_counter() - t_load:.1f}s", flush=True)

base_answers = []
for step, example in enumerate(examples, start=1):
    pubid = example.get("pubid")
    print(f"[base] Example {step}/{n_examples} | pubid={pubid} — generating …", flush=True)
    t0 = time.perf_counter()
    prompt = build_prompt(example)
    print(f"[base]   prompt: {len(prompt)} chars", flush=True)
    answer = generate_answer(base_tokenizer, base_model, prompt)
    base_answers.append(answer)
    dt = time.perf_counter() - t0
    preview = answer if len(answer) <= 200 else answer[:200] + "…"
    print(f"[base]   done in {dt:.1f}s", flush=True)
    if torch.cuda.is_available():
        print(f"[base]   GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GiB", flush=True)
    print(f"[base]   answer preview: {preview!r}", flush=True)

release_model(base_model)
print(f"[base] Finished all {n_examples} examples.", flush=True)

## Run Trained Model

This loads the trained model repo returned by Gradients. If the repo is a LoRA adapter, it is merged with the original base model before generation.

In [ ]:
if "TRAINED_MODEL_REPO" not in globals() or not TRAINED_MODEL_REPO:
    TRAINED_MODEL_REPO = input("Paste the trained_model_repository from Gradients: ").strip()

n_examples = len(examples)
print(f"[trained] Loading {TRAINED_MODEL_REPO!r} ({n_examples} examples) …", flush=True)
t_load = time.perf_counter()
trained_tokenizer, trained_model = load_model(TRAINED_MODEL_REPO, base_model_id=MODEL_ID)
print(f"[trained] Load finished in {time.perf_counter() - t_load:.1f}s", flush=True)

trained_answers = []
for step, example in enumerate(examples, start=1):
    pubid = example.get("pubid")
    print(f"[trained] Example {step}/{n_examples} | pubid={pubid} — generating …", flush=True)
    t0 = time.perf_counter()
    prompt = build_prompt(example)
    print(f"[trained]   prompt: {len(prompt)} chars", flush=True)
    answer = generate_answer(trained_tokenizer, trained_model, prompt)
    trained_answers.append(answer)
    dt = time.perf_counter() - t0
    preview = answer if len(answer) <= 200 else answer[:200] + "…"
    print(f"[trained]   done in {dt:.1f}s", flush=True)
    if torch.cuda.is_available():
        print(f"[trained]   GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GiB", flush=True)
    print(f"[trained]   answer preview: {preview!r}", flush=True)

release_model(trained_model)
print(f"[trained] Finished all {n_examples} examples.", flush=True)

## Compare Results

The table shows the expected answer from the held-out test set alongside the base model and trained model outputs.

In [ ]:
def clean_text(value):
    return (value or "").strip()


comparison_rows = []
for index, example in enumerate(examples, start=1):
    prompt = build_prompt(example)
    question = prompt.removesuffix("Answer:").strip()
    comparison_rows.append(
        {
            "example": index,
            "pubid": example.get("pubid"),
            "question": question,
            "expected": example.get("output"),
            "base_model": base_answers[index - 1],
            "trained_model": trained_answers[index - 1],
        }
    )

for row in comparison_rows:
    display(Markdown(f"""
---

## Example {row['example']} | PubMed ID: {row['pubid']}

### Question

{clean_text(row['question'])}

### Expected Answer

{clean_text(row['expected'])}

### Trained Model Answer

{clean_text(row['trained_model'])}

### Base Model Answer

{clean_text(row['base_model'])}
"""))